<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install all required libraries for downloading market data, running statistical tests, and plotting results.

In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn statsmodels

## Imports and setup

numpy and pandas handle numerical arrays and tabular data. statsmodels provides the cointegration test (coint), ordinary least squares regression (OLS), and rolling regression tools we use to measure and track the relationship between stock pairs. yfinance downloads historical price data from Yahoo Finance. seaborn and matplotlib produce the heatmap and time series plots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn
import statsmodels.api as sm
import yfinance as yf
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import coint

## Download price data for candidate pairs

We download one year of daily closing prices for five large-cap tech stocks. Starting with a small, economically related group reflects the professional approach described in the post: pick companies that share a real business connection first, then let statistics confirm or reject the relationship.

In [ ]:
symbol_list = ['META', 'AMZN', 'AAPL', 'NFLX', 'GOOG']
data = yf.download(
    symbol_list,
    start='2014-01-01',
    end='2015-01-01'
)
data = data.Close

Using only closing prices keeps the analysis simple and focused on end-of-day valuations, which is the standard starting point for pairs trading research. Notice we deliberately chose a narrow universe of five stocks rather than scanning hundreds. As the post explains, blindly testing every combination inflates false positives because you're running too many comparisons without an economic thesis.

## Test all pairs for cointegration

This function runs the Engle-Granger cointegration test on every unique pair in our universe and collects pairs whose p-value falls below 0.05, meaning there is statistical evidence that their price relationship is mean-reverting rather than random.

In [ ]:
def find_cointegrated_pairs(data):
    n = data.shape[1]
    score_matrix = np.zeros((n, n))
    pvalue_matrix = np.ones((n, n))
    keys = data.keys()
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            S1 = data[keys[i]]
            S2 = data[keys[j]]
            result = coint(S1, S2)
            score = result[0]
            pvalue = result[1]
            score_matrix[i, j] = score
            pvalue_matrix[i, j] = pvalue
            if pvalue < 0.05:
                pairs.append((keys[i], keys[j]))
    return score_matrix, pvalue_matrix, pairs

Cointegration is the core statistical concept behind pairs trading. Unlike correlation, which only measures whether two stocks move in the same direction, cointegration tests whether the gap between two price series tends to revert to a stable level over time. The p-value matrix we build here will let us visually compare every pair at once in the next step.

Run the cointegration scan across all pairs and visualize which relationships pass the significance threshold.

In [ ]:
scores, pvalues, pairs = find_cointegrated_pairs(data)

In [ ]:
seaborn.heatmap(
    pvalues,
    xticklabels=symbol_list,
    yticklabels=symbol_list,
    cmap='RdYlGn_r',
    mask=(pvalues >= 0.05)
)
plt.show()

The mask hides every pair with a p-value at or above 0.05, so only statistically significant relationships appear on the heatmap. This gives us a quick visual filter. With five stocks we have only ten unique pairs, but in a larger universe this heatmap becomes essential for spotting candidates without scrolling through tables of numbers.

## Compute the spread and hedge ratio

We isolate the AMZN-AAPL pair and use ordinary least squares regression to estimate the hedge ratio, which tells us how many dollars of AAPL to hold per dollar of AMZN so the combined position is balanced. The spread is the residual: the part of AAPL's price that AMZN's price cannot explain.

In [ ]:
S1 = data['AMZN']
S2 = data['AAPL']

In [ ]:
S1 = sm.add_constant(S1)
results = sm.OLS(S2, S1).fit()
S1 = S1.AMZN
b = results.params['AMZN']
spread = S2 - b * S1

The hedge ratio b is what makes this a market-neutral strategy. Without it, we would be exposed to the overall direction of the tech sector. By subtracting b times AMZN from AAPL, we isolate only the relative mispricing between the two stocks. If the cointegration relationship holds, this spread should fluctuate around a stable mean.

Plot the spread over time with a horizontal line at its mean to see whether it behaves like a mean-reverting series.

In [ ]:
spread.plot()
plt.axhline(spread.mean(), color='black')
plt.legend(['Spread'])
plt.show()

Visually checking the spread is a sanity step before trading. We want to see the spread crossing its mean repeatedly. A spread that trends away from the mean suggests the cointegration relationship may be breaking down, which is exactly the kind of warning that separates careful analysis from blind backtesting.

Define a z-score function that standardizes the spread, then plot it with threshold lines at +1 and -1 to identify trade entry points.

In [ ]:
def zscore(series):
    return (series - series.mean()) / np.std(series)

In [ ]:
zscore(spread).plot()
plt.axhline(zscore(spread).mean(), color='black')
plt.axhline(1.0, color='red', linestyle='--')
plt.axhline(-1.0, color='green', linestyle='--')
plt.legend(['Spread z-score', 'Mean', '+1', '-1'])
plt.show()

The z-score converts the spread into units of standard deviation, which makes it easy to set universal entry thresholds regardless of the dollar scale of the spread. When the z-score drops below -1, the spread is unusually cheap and we go long. When it rises above +1, the spread is unusually expensive and we go short. These thresholds are a simple starting point; professionals often optimize them or add exit rules at the mean.

## Build and evaluate the trading strategy

Combine the z-score signal and the dollar spread into a single DataFrame, then assign trade direction: go long the spread when the z-score is at or below -1, and short it when the z-score is at or above +1.

In [ ]:
trades = pd.concat([zscore(spread), S2 - b * S1], axis=1)
trades.columns = ["signal", "position"]

In [ ]:
trades["side"] = 0.0
trades.loc[trades.signal <= -1, "side"] = 1
trades.loc[trades.signal >= 1, "side"] = -1

The side column holds +1, -1, or 0, representing long, short, or flat. We multiply the daily percentage change of the spread by this side value to get our strategy return each day. This is a simplified position model that assumes we enter and exit at closing prices with no transaction costs, which is fine for learning but would need refinement before live trading.

Calculate daily strategy returns by multiplying the spread's percentage change by our position direction, then plot cumulative returns to see how the strategy performed.

In [ ]:
returns = trades.position.pct_change() * trades.side
returns.cumsum().plot()
plt.show()

As the post warns, this strategy loses money over the test period. That result is intentional and instructive. A passing cointegration test on historical data does not guarantee profitable trading. The z-score thresholds use the full sample mean and standard deviation, which introduces lookahead bias since we would not know those values in real time. Addressing that bias with rolling windows, adding transaction costs, and testing on out-of-sample data are the natural next steps that separate a learning exercise from a deployable strategy.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.